# Session 2: Prompt Engineering in Practice

## Objectives
- Master zero-shot, few-shot, and chain-of-thought prompting
- Design effective system prompts
- Use prompt templates for reusable patterns
- Compare prompting strategies on real tasks

**Duration:** 40 minutes | **Level:** Medium

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "rnj-1-instruct")

def chat(user_message, system_message="You are a helpful assistant.", model=MODEL, temperature=0.7):
    """Simple helper to call the Chat Completions API."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

print(f"Setup complete! Model: {MODEL}")

Setup complete! Model: rnj-1-instruct


## 1. Zero-Shot Prompting

**Zero-shot** = Give the model a task with NO examples.
The model relies entirely on its training knowledge.

Works well for simple, well-defined tasks.

In [2]:
# Zero-shot: Classify sentiment without any examples
prompt = """Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The food was absolutely delicious and the service was impeccable!"

Sentiment:"""

result = chat(prompt, temperature=0)
print(result)

POSITIVE

The review expresses strong positive sentiments about both the food quality ("absolutely delicious") and the service quality ("impeccable"). The use of superlative adjectives like "absolutely" and "impeccable" amplifies the positivity, indicating a highly favorable experience. There are no negative or neutral elements mentioned in the review.

Answer:
\boxed{POSITIVE}


In [3]:
# Zero-shot: Extract information
prompt = """Extract the person's name, age, and occupation from this text.

Text: "Dr. Sarah Chen, a 42-year-old neuroscientist at MIT, published groundbreaking research."

Output:"""

result = chat(prompt, temperature=0)
print(result)

Name: Dr. Sarah Chen
Age: 42
Occupation: Neuroscientist


## 2. Few-Shot Prompting

**Few-shot** = Provide a few examples before the actual task.
This helps the model understand:
- The exact format you want
- Edge cases and expected behavior
- The "style" of the output

In [4]:
# Few-shot: Sentiment classification with examples
prompt = """Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "Great product, works perfectly!"
Sentiment: POSITIVE

Review: "Terrible quality, broke after one day."
Sentiment: NEGATIVE

Review: "It's okay, nothing special."
Sentiment: NEUTRAL

Review: "The battery life could be better but the camera is amazing."
Sentiment:"""

result = chat(prompt, temperature=0)
print(result)

NEUTRAL

The review contains both positive and negative aspects. The statement about the camera being amazing indicates a positive sentiment, while the comment on the battery life suggests a negative sentiment. Since there are equal parts of both sentiments without a clear overall leaning towards one, the overall sentiment is considered NEUTRAL.


In [5]:
# Few-shot: Custom text classification
# The examples teach the model YOUR specific categories
prompt = """Classify the support ticket into one of these categories: BILLING, TECHNICAL, ACCOUNT, OTHER.

Ticket: "I was charged twice for my subscription"
Category: BILLING

Ticket: "The app keeps crashing on my phone"
Category: TECHNICAL

Ticket: "I need to reset my password"
Category: ACCOUNT

Ticket: "My dashboard is not loading and shows a 500 error"
Category:"""

result = chat(prompt, temperature=0)
print(result)

TECHNICAL


## 3. Chain-of-Thought (CoT) Prompting

**Chain-of-Thought** = Ask the model to "think step by step" before answering.

This dramatically improves performance on:
- Math and logic problems
- Complex reasoning tasks
- Multi-step analysis

Two approaches:
1. **Simple CoT**: Add "Think step by step" to the prompt
2. **Few-shot CoT**: Show examples with reasoning steps

In [6]:
# Without CoT â€” the model might get this wrong
prompt_no_cot = """A store has 15 apples. It sells 8 in the morning and receives a shipment of 12.
Then it sells 6 more in the afternoon. How many apples are left?

Answer:"""

print("Without CoT:")
print(chat(prompt_no_cot, temperature=0))

# With CoT â€” asking the model to reason step by step
prompt_cot = """A store has 15 apples. It sells 8 in the morning and receives a shipment of 12.
Then it sells 6 more in the afternoon. How many apples are left?

Think step by step, then give the final answer."""

print("\nWith CoT:")
print(chat(prompt_cot, temperature=0))

Without CoT:
Let's calculate step by step:

1. The store starts with **15 apples**.

2. In the morning, it sells **8 apples**:
   $$
   15 - 8 = 7 \text{ apples left}
   $$

3. Then it receives a shipment of **12 apples**:
   $$
   7 + 12 = 19 \text{ apples now}
   $$

4. In the afternoon, it sells **6 more apples**:
   $$
   19 - 6 = 13 \text{ apples left}
   $$

### Final Answer:
$$
\boxed{13}
$$

With CoT:
Let's break down the problem step by step:

1. The store starts with 15 apples.
2. It sells 8 apples in the morning, so we subtract 8 from 15: 15 - 8 = 7 apples left.
3. Then it receives a shipment of 12 apples, so we add 12 to 7: 7 + 12 = 19 apples now.
4. Finally, it sells 6 more apples in the afternoon, so we subtract 6 from 19: 19 - 6 = 13 apples left.

So, there are \boxed{13} apples left.


In [7]:
# Few-shot CoT: Show the reasoning process in examples
prompt = """Determine if the conclusion follows from the premise. Show your reasoning.

Premise: "All mammals are warm-blooded. Whales are mammals."
Reasoning: Since all mammals are warm-blooded and whales are mammals, whales must be warm-blooded.
Conclusion follows: Yes

Premise: "Some birds can fly. Penguins are birds."
Reasoning: The premise says SOME birds can fly, not ALL. Penguins being birds doesn't guarantee they can fly.
Conclusion follows: No

Premise: "All students who passed the exam studied hard. John studied hard."
Reasoning:"""

result = chat(prompt, temperature=0)
print(result)

Premise: "All students who passed the exam studied hard. John studied hard."

This is a logical fallacy known as the converse error or inverse error. Just because all students who passed studied hard doesn't mean that everyone who studied hard passed.

The correct reasoning would be:
- All A are B (All students who passed are hard workers)
- C is B (John is a hard worker)
- This does NOT necessarily mean C is A (John did not necessarily pass)

Conclusion follows: No


## 4. System Prompt Design Patterns

The **system prompt** is your most powerful tool. Common patterns:

1. **Role assignment**: "You are a [role]..."
2. **Constraints**: "Only respond with...", "Never..."
3. **Format specification**: "Respond in bullet points"
4. **Behavior rules**: "If you don't know, say so"

In [8]:
# Pattern 1: Expert role with constraints
system_prompt = """You are a senior Python code reviewer.
- Review the code for bugs, style issues, and improvements
- Rate the code quality: GOOD, NEEDS_IMPROVEMENT, or POOR
- Keep feedback concise (max 3 bullet points)
- Always suggest at least one improvement"""

code_to_review = """
def calc(x,y,op):
    if op == 'add': return x+y
    if op == 'sub': return x-y
    if op == 'mul': return x*y
    if op == 'div': return x/y
"""

result = chat(code_to_review, system_message=system_prompt, temperature=0)
print(result)

GOOD

The code is correct and performs the intended operations. However, here are a few suggestions for improvement:

- Add error handling for division by zero.
- Consider using a dictionary to map operation names to functions for better scalability.
- Add type hints for better code readability and maintainability.

Here's an improved version of the code:

```python
def calc(x: float, y: float, op: str) -> float:
    """
    Perform arithmetic operations on two numbers.

    Args:
        x (float): The first number.
        y (float): The second number.
        op (str): The operation to perform. Can be 'add', 'sub', 'mul', or 'div'.

    Returns:
        float: The result of the operation.

    Raises:
        ValueError: If an invalid operation is provided.
        ZeroDivisionError: If division by zero is attempted.
    """
    operations = {
        'add': lambda a, b: a + b,
        'sub': lambda a, b: a - b,
        'mul': lambda a, b: a * b,
        'div': lambda a, b: a / b
  

In [9]:
# Pattern 2: Structured output instruction
system_prompt = """You are a text analysis assistant.
For every input text, respond with EXACTLY this format:

TOPIC: [main topic]
TONE: [formal/informal/neutral]
KEY_POINTS: [comma-separated key points]
WORD_COUNT: [approximate word count of input]"""

text = """Artificial intelligence has been transforming industries at an unprecedented rate.
From healthcare to finance, AI-powered solutions are improving efficiency and accuracy.
However, ethical concerns around bias and privacy remain significant challenges."""

result = chat(text, system_message=system_prompt, temperature=0)
print(result)

TOPIC: Artificial Intelligence in Industries

TONE: Informal

KEY_POINTS: 
AI transformation of industries,
Improvement in efficiency and accuracy,
Ethical concerns about bias and privacy.

WORD_COUNT: 35


## 5. Prompt Templates

Use Python f-strings to create reusable prompt templates.
This makes your prompts maintainable and parameterized.

In [10]:
# Reusable prompt template for different tasks
def analyze_text(text, analysis_type):
    """Analyze text with a specified analysis type."""
    template = f"""Perform {analysis_type} analysis on the following text.
Be concise and specific in your analysis.

Text: \"{text}\"

Analysis:"""
    return chat(template, temperature=0)

sample_text = "The new policy will increase taxes for high earners while providing relief for small businesses."

# Use the same template for different analysis types
print("=== Sentiment Analysis ===")
print(analyze_text(sample_text, "sentiment"))

print("\n=== Bias Analysis ===")
print(analyze_text(sample_text, "bias"))

=== Sentiment Analysis ===
Sentiment Analysis:

The sentiment of this text is mixed, with both positive and negative aspects. The statement presents a balanced view by acknowledging the burden on high earners through increased taxes but also highlights the benefits for small businesses in terms of tax relief.

Positive Aspects:
- Relief for small businesses

Negative Aspects:
- Increased taxes for high earners

=== Bias Analysis ===
**Bias Analysis of the Text**

1. **Targeted Tax Increase**: The statement specifically mentions "high earners," indicating a focus on taxing a particular demographic group, which could be seen as targeting or singling out this group.

2. **Relief for Small Businesses**: By contrast, the text highlights relief for small businesses without specifying criteria, potentially implying that these businesses are more deserving of tax breaks than high earners.

3. **Implied Wealth Distribution**: The policy's framing suggests a redistribution of wealth from one seg

## Exercise: Comparing Prompting Strategies

Build a product review classifier that categorizes reviews and extracts key information.
Try **zero-shot**, **few-shot**, and **CoT** approaches and compare results.

In [11]:
# Test review for all strategies
test_review = """The laptop arrived quickly but the packaging was damaged.
The device itself works fine â€” fast processor, beautiful display.
However, the battery barely lasts 3 hours which is disappointing for the price.
Customer support was helpful when I reported the packaging issue."""

# Strategy 1: Zero-shot
zero_shot_prompt = f"""Analyze this product review. Provide: overall sentiment, 
pros, cons, and a rating out of 5.

Review: \"{test_review}\""""

print("=== Zero-Shot ===")
print(chat(zero_shot_prompt, temperature=0))

=== Zero-Shot ===
**Overall Sentiment:** Mixed

**Pros:**
- Quick delivery of the laptop
- Fast processor
- Beautiful display
- Helpful customer support

**Cons:**
- Damaged packaging upon arrival
- Short battery life (barely lasts 3 hours)

**Rating out of 5:** 3.5/5


In [12]:
# Strategy 2: Few-shot
few_shot_prompt = f"""Analyze product reviews in this exact format:

Review: "Amazing phone, great camera, but too expensive."
Sentiment: MIXED
Pros: great camera
Cons: too expensive
Rating: 3.5/5

Review: "Perfect headphones, noise cancellation is incredible, very comfortable."
Sentiment: POSITIVE
Pros: noise cancellation, comfort
Cons: none mentioned
Rating: 5/5

Review: \"{test_review}\"
Sentiment:"""

print("=== Few-Shot ===")
print(chat(few_shot_prompt, temperature=0))

=== Few-Shot ===
Review: "Amazing phone, great camera, but too expensive."
Sentiment: MIXED
Pros: great camera
Cons: too expensive
Rating: 3.5/5

Review: "Perfect headphones, noise cancellation is incredible, very comfortable."
Sentiment: POSITIVE
Pros: noise cancellation, comfort
Cons: none mentioned
Rating: 5/5

Review: "The laptop arrived quickly but the packaging was damaged.
The device itself works fine â€” fast processor, beautiful display.
However, the battery barely lasts 3 hours which is disappointing for the price.
Customer service was helpful when I reported the packaging issue."
Sentiment: MIXED
Pros: quick delivery, fast processor, beautiful display, helpful customer support
Cons: damaged packaging, short battery life
Rating: 4/5


In [13]:
# Strategy 3: Chain-of-Thought
cot_prompt = f"""Analyze this product review step by step:
1. First, identify all positive points mentioned
2. Then, identify all negative points mentioned
3. Consider the overall tone and weight of positive vs negative
4. Assign a final sentiment and rating

Review: \"{test_review}\"

Step-by-step analysis:"""

print("=== Chain-of-Thought ===")
print(chat(cot_prompt, temperature=0))

=== Chain-of-Thought ===
1. Positive points:
   - Laptop arrived quickly (delivery speed)
   - Device works fine (functionality)
   - Fast processor (performance)
   - Beautiful display (visual quality)
   - Customer support was helpful (service)

2. Negative points:
   - Packaging was damaged (shipping issue)
   - Battery barely lasts 3 hours (battery life)
   - Disappointing for the price (value perception)

3. Overall tone analysis:
   - Positive aspects are balanced with negative ones
   - The reviewer acknowledges both strengths and weaknesses
   - Specific technical praise is mixed with practical concerns

4. Final sentiment and rating:
   - Sentiment: Mixed/Neutral
   - Rating: 3.5/5
   - Justification: While there are clear positives about the laptop's performance and customer service, the significant battery life issue (which affects portability) and packaging damage (a shipping failure) create substantial concerns that outweigh the positive aspects for many users.

The review

## Summary

| Strategy | When to Use |
|----------|------------|
| **Zero-shot** | Simple, well-defined tasks |
| **Few-shot** | When you need specific output format or behavior |
| **Chain-of-Thought** | Complex reasoning, math, multi-step analysis |

**Key takeaways:**
- Be specific and explicit in your prompts
- System prompts shape behavior; user prompts provide the task
- Few-shot examples are powerful for format control
- CoT improves reasoning accuracy significantly

**Next session:** Getting structured (JSON) outputs from LLMs!